# Day 3 — Medallion Architecture: Bronze → Silver → Gold (Delta Lake)

This hands-on builds a **batch medallion pipeline** on **Azure Data Lake Gen2** using **ABFS** (`abfss://...`) only — the same pattern as Day 1 and Day 2 (**OAuth via `spark.conf`**, **`%run ./02-Mount-Azure-Data-Lake`**). No DBFS mount, no File Store for production paths.

**Reference labs** (external courseware under `.../databricks/`, patterns adapted here to **ABFS**):

1. **`azure-databricks-tata-technologies-main/Labs/08-Reading Writing to Delta Tables.ipynb`** — Parquet→Delta, **`DeltaTable.forPath`**, **`partitionBy`**, SQL **`MERGE`**.
2. **`azure-databricks-tata-technologies-main/Labs/09-DeltaTableVersioning.ipynb`** — **`DESCRIBE HISTORY`**, **`versionAsOf` / `timestampAsOf`**, restore concepts.
3. **`azure-databricks-tata-technologies-main/Labs/01-Databricks-datalake-spark.ipynb`** — object-store paths, **SQL vs DataFrame** same query, **`explain()`**.

**Prerequisites:** `data/2010-summary.csv` and `data/json/2015-summary.json` under your `base_path` (same container layout as earlier days). Optional: `data/parquet` if you built Parquet in Day 1.

## 1 — Configure storage (run mount helper first)

Run **`02-Mount-Azure-Data-Lake`** once per cluster session *or* use the next cell (`%run`).

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
# Paths from %run: base_path, containerName, adlsAccountName
BASE_PATH = base_path
OUTPUT_PATH = f"abfss://{containerName}@{adlsAccountName}.dfs.core.windows.net/data/OUTPUT"

# Medallion layout under the same container (adjust only if your convention differs)
DAY03_ROOT = BASE_PATH + "/day03"
BRONZE_PATH = DAY03_ROOT + "/bronze/flights_raw"
SILVER_PATH = DAY03_ROOT + "/silver/flights_clean"
GOLD_PATH = DAY03_ROOT + "/gold/flights_by_destination"
GOLD_PARTITIONED_PATH = DAY03_ROOT + "/gold/flights_by_dest_partitioned"
BRONZE_FROM_PARQUET_PATH = DAY03_ROOT + "/bronze/flights_from_parquet"

# Source files (same as Day 1 / Day 2 layout)
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
# Optional: created in Day 1 notebook (CSV → Parquet)
PARQUET_SOURCE = BASE_PATH + "/parquet"

print("BASE_PATH:", BASE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("GOLD_PARTITIONED_PATH:", GOLD_PARTITIONED_PATH)
print("PARQUET_SOURCE (optional):", PARQUET_SOURCE)

## 2 — Medallion layers (quick recap)

| Layer | Purpose | Typical write mode |
|-------|---------|-------------------|
| **Bronze** | Raw ingest + audit columns | `append` (or first-time `overwrite`) |
| **Silver** | Clean, typed, deduped, conformed | `overwrite` or `merge` (see Day 3 advanced notebook) |
| **Gold** | Business aggregates / marts | `overwrite` |

**Delta Lake** gives you ACID writes, schema enforcement, time travel, and `MERGE` for upserts — ideal for Silver and Gold.

## 3 — Bronze: ingest CSV with metadata

**Bronze** keeps data close to the source and adds **lineage** (`ingestion_ts`, `source_file`).

In [ ]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

# Unity Catalog: input_file_name() is not supported — use _metadata.file_path (see UC docs).
# coalesce(..., lit(SOURCE_CSV)) covers single-file reads if metadata is null on some runtimes.
bronze_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
    .withColumn("source_system", lit("day3_lab_csv"))
)

bronze_df.printSchema()
bronze_df.show(5, truncate=False)

(
    bronze_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BRONZE_PATH)
)

print("Bronze written to:", BRONZE_PATH)

## 4 — Silver: clean, type, dedupe, surrogate key

**Silver** applies **data-quality rules** and a stable **`route_key`** (for merges in the advanced notebook).

In [ ]:
from pyspark.sql.functions import col, sha2, concat_ws

silver_df = (
    spark.read.format("delta")
    .load(BRONZE_PATH)
    .select(
        "DEST_COUNTRY_NAME",
        "ORIGIN_COUNTRY_NAME",
        "count",
        "ingestion_ts",
        "source_file",
        "source_system",
    )
    .filter(col("DEST_COUNTRY_NAME").isNotNull() & col("ORIGIN_COUNTRY_NAME").isNotNull())
    .filter(col("count").isNotNull())
    .withColumn("count", col("count").cast("long"))
    .withColumn(
        "route_key",
        sha2(concat_ws("|", col("DEST_COUNTRY_NAME"), col("ORIGIN_COUNTRY_NAME")), 256),
    )
    .dropDuplicates(["route_key"])
)

silver_df.printSchema()
silver_df.show(5, truncate=False)

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print("Silver written to:", SILVER_PATH)

## 5 — Gold: curated aggregate for analytics

**Gold** exposes a **consumption-ready** dataset (here: total flights by destination).

In [ ]:
from pyspark.sql.functions import sum as spark_sum, col

gold_df = (
    spark.read.format("delta")
    .load(SILVER_PATH)
    .groupBy("DEST_COUNTRY_NAME")
    .agg(spark_sum("count").alias("total_flights"))
    .orderBy(col("total_flights").desc())
)

gold_df.show(15, truncate=False)

(
    gold_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH)
)

print("Gold written to:", GOLD_PATH)

## 5b — Extra example A: `DeltaTable.forPath` → DataFrame (Lab *08* pattern)

Read a Delta table through the **Delta** API (`toDF()`), same idea as `DeltaTable.forPath(spark, path)` in the external *Reading Writing Delta* lab.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

DeltaTable.forPath(spark, GOLD_PATH).toDF().orderBy(col("total_flights").desc()).show(5, truncate=False)

## 5c — Extra example B: **Partitioned** Gold write (`partitionBy`) — Lab *08* pattern

External lab writes Delta with **`partitionBy`** (e.g. by year). Here we partition by the **first letter** of destination (demo only — choose real partition keys in production).

In [ ]:
from pyspark.sql.functions import substring, upper as upper_col, col

gold_part = gold_df.withColumn(
    "dest_letter",
    upper_col(substring(col("DEST_COUNTRY_NAME"), 1, 1)),
)

(
    gold_part.write.format("delta")
    .partitionBy("dest_letter")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PARTITIONED_PATH)
)

print("Partitioned Gold Delta at:", GOLD_PARTITIONED_PATH)
spark.sql(f"DESCRIBE DETAIL delta.`{GOLD_PARTITIONED_PATH}`").show(truncate=False)

## 5d — Extra example C: **Parquet → Delta** into Bronze (Lab *08* pattern)

If **`PARQUET_SOURCE`** exists (e.g. from Day 1), land a **Bronze** copy from Parquet with metadata — mirrors *read Parquet, add column, write Delta* in the external lab.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

try:
    pq_df = spark.read.format("parquet").load(PARQUET_SOURCE)
    pq_bronze = (
        pq_df.withColumn("ingestion_ts", current_timestamp())
        .withColumn("source_system", lit("day3_parquet_to_delta"))
    )
    (
        pq_bronze.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(BRONZE_FROM_PARQUET_PATH)
    )
    print("Bronze (from Parquet) written to:", BRONZE_FROM_PARQUET_PATH)
    pq_bronze.printSchema()
except Exception as e:
    print("Skipped Parquet→Delta example (path missing or unreadable):", type(e).__name__, e)

## 5e — Extra example D: **Same query** — SQL vs DataFrame + `explain()` (Lab *01* pattern)

The external *datalake* lab compares **physical plans** for equivalent SQL and DataFrame aggregations.

In [ ]:
from pyspark.sql.functions import sum as spark_sum

spark.read.format("delta").load(SILVER_PATH).createOrReplaceTempView("silver_flights_explain")

sql_way = spark.sql(
    '''
    SELECT DEST_COUNTRY_NAME, SUM(count) AS total_flights
    FROM silver_flights_explain
    GROUP BY DEST_COUNTRY_NAME
    '''
)

df_way = (
    spark.read.format("delta")
    .load(SILVER_PATH)
    .groupBy("DEST_COUNTRY_NAME")
    .agg(spark_sum("count").alias("total_flights"))
)

print("--- SQL physical plan ---")
sql_way.explain("formatted")
print("--- DataFrame physical plan ---")
df_way.explain("formatted")

## 6 — Validation: row counts and Delta detail

Quick checks: Bronze ≥ Silver row counts (Silver drops nulls/dupes), Gold has fewer rows than Silver.

In [ ]:
bronze_n = spark.read.format("delta").load(BRONZE_PATH).count()
silver_n = spark.read.format("delta").load(SILVER_PATH).count()
gold_n = spark.read.format("delta").load(GOLD_PATH).count()

print("Bronze rows:", bronze_n)
print("Silver rows:", silver_n)
print("Gold rows (destinations):", gold_n)

spark.sql(f"DESCRIBE DETAIL delta.`{BRONZE_PATH}`").show(truncate=False)

## 7 — SQL over Delta (optional)

Use **backticks** around ABFSS paths in SQL.

In [ ]:
spark.read.format("delta").load(SILVER_PATH).createOrReplaceTempView("silver_flights")

spark.sql(
    '''
    SELECT DEST_COUNTRY_NAME, SUM(count) AS total_flights
    FROM silver_flights
    GROUP BY DEST_COUNTRY_NAME
    ORDER BY total_flights DESC
    LIMIT 10
    '''
).show(truncate=False)

## Next steps

Open **`03-Day3-Delta-Lake-Advanced`** for **table history**, **time travel**, **`MERGE`**, **schema evolution**, and **`OPTIMIZE`** (Databricks Delta).

---

### Extra topics (self-study)

* **Auto Loader** (`cloudFiles`) for incremental file ingestion into Bronze — see Databricks docs and *Structured Streaming* labs in external courseware.
* **Unity Catalog** — register these paths as managed/external tables for governance (instructor demo).
* **Expectations / Delta Live Tables** — declarative quality + dependencies (overview only in many courses).